# Credit Card Fraud Detection — Phase 2: Preprocessing & Feature Engineering

**Phase 1 notebook:** `notebooks/01_data_audit.ipynb`  
**Dataset:** `data/creditcard.csv` (read-only)  
**Rules:**
- No ML model training in this notebook.
- No SMOTE in this notebook.
- Do not overwrite `creditcard.csv`.
- Scaler parameters must be fitted on training data only.
- RANDOM_SEED = 42 throughout.

---

## 1. Setup & Imports

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Add project root to path so src.preprocessing is importable ──────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── Import our reusable preprocessing module ──────────────────────────────────
from src.preprocessing import (
    load_raw_data,
    audit_duplicates,
    drop_exact_duplicates,
    engineer_features,
    split_features_target,
    make_train_test_split,
    fit_scaler,
    apply_scaler,
    get_outlier_thresholds,
    apply_winsorisation,
    build_full_pipeline,
    RANDOM_SEED, TARGET_COL, TEST_SIZE, COLS_TO_SCALE,
)

np.random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA_PATH = os.path.join(PROJECT_ROOT, "data", "creditcard.csv")
print(f"Project root  : {PROJECT_ROOT}")
print(f"Data path     : {DATA_PATH}")
print(f"File exists   : {os.path.exists(DATA_PATH)}")
print(f"RANDOM_SEED   : {RANDOM_SEED}")
print(f"TEST_SIZE     : {TEST_SIZE}")
print(f"COLS_TO_SCALE : {COLS_TO_SCALE}")


---
## 2. Load Data

In [ ]:
# Load the raw CSV — read-only; never modified in any phase
df_raw = load_raw_data(DATA_PATH)

ORIGINAL_SHAPE = df_raw.shape
file_size_mb   = os.path.getsize(DATA_PATH) / 1e6

print(f"Original dataset shape : {ORIGINAL_SHAPE[0]:,} rows x {ORIGINAL_SHAPE[1]} columns")
print(f"File size on disk      : {file_size_mb:.1f} MB")
print()

# Integrity assertions — verify the CSV matches Phase 1 expectations
assert df_raw.shape[1] == 31,                      "Unexpected column count — CSV may be altered."
assert set(df_raw["Class"].unique()) == {0, 1},    "Unexpected Class values."
assert df_raw.isnull().sum().sum() == 0,           "Unexpected missing values."
assert df_raw.select_dtypes(include="object").shape[1] == 0, "Unexpected non-numeric columns."

print("Dataset integrity checks: PASSED")
print(f"  Columns : {ORIGINAL_SHAPE[1]}  (30 features + 1 target)")
print(f"  Missing : 0")
print(f"  Class   : binary {{0, 1}}")


---
## 3. Apply Phase 1 Decisions

In [ ]:
print("Phase 1 Recommendations — Applied in Phase 2")
print("=" * 55)
print()
print("  [1] Drop exact duplicates BEFORE train/test split.")
print("  [2] Stratified split: test_size=0.20, random_state=42.")
print("  [3] RobustScaler on Amount, Time, Log_Amount, Hour.")
print("  [4] V1-V28 NOT rescaled (already PCA-normalised).")
print("  [5] SMOTE deferred to model training phase (Phase 3).")
print("  [6] Outlier treatment: document thresholds, preserve values by default.")
print("  [7] Engineer Log_Amount (corrects skew) and Hour (diurnal signal).")
print("  [8] Primary eval metric in Phase 3: Average Precision (PR-AUC).")


---
## 4. Duplicate Handling

In [ ]:
print("=" * 60)
print("DUPLICATE AUDIT (before removal)")
print("=" * 60)

dup_report = audit_duplicates(df_raw)
n_orig     = dup_report["n_total_rows"]
n_dup      = dup_report["n_duplicate_rows"]
pct_dup    = dup_report["pct_duplicate"]

print(f"  Total rows               : {n_orig:,}")
print(f"  Duplicate rows (non-1st) : {n_dup:,}  ({pct_dup:.4f}%)")
print()
print("  Class breakdown in duplicated rows:")
for cls, cnt in dup_report["dup_class_breakdown"].items():
    label = "Genuine" if cls == 0 else "Fraud"
    pct   = cnt / (dup_report["n_duplicate_rows"] * 2 or 1) * 100
    print(f"    Class {cls} ({label}): {cnt:,}")

conflicts = dup_report["conflicting_pairs"]
print()
if conflicts.empty:
    print("  Conflicting-label duplicates : None detected.")
    print("  Decision: Safe to remove exact duplicates (keep first occurrence).")
else:
    print(f"  WARNING: {len(conflicts)} rows have identical features but DIFFERENT Class labels.")
    print("  These will NOT be silently removed. See below:")
    print(conflicts.head(10).to_string())


In [ ]:
# Drop exact duplicates — BEFORE any train/test split
print("Removing exact duplicates (keep='first') ...")
df_clean = drop_exact_duplicates(df_raw, keep="first", verbose=False)

n_removed = n_orig - len(df_clean)
print(f"  Rows before dedup : {n_orig:,}")
print(f"  Rows removed      : {n_removed:,}")
print(f"  Rows remaining    : {len(df_clean):,}")
print()

# Verify original CSV was NOT modified
df_verify = pd.read_csv(DATA_PATH)
assert df_verify.shape == ORIGINAL_SHAPE, "CRITICAL: Original CSV was modified!"
del df_verify
print("Original CSV integrity check : VERIFIED (untouched)")


---
## 5. Target Separation (X / y)

In [ ]:
# Engineer features on the deduplicated dataset
df_feat = engineer_features(df_clean)

print("Engineered features added to deduplicated dataset:")
print(f"  Log_Amount : log1p(Amount)             — corrects right-skew")
print(f"  Hour       : (Time % 86400) / 3600     — captures diurnal pattern")
print(f"  Dataset shape before: {df_clean.shape}")
print(f"  Dataset shape after : {df_feat.shape}")
print()

# Separate features from target
X, y = split_features_target(df_feat, target_col="Class")

print(f"X shape : {X.shape}  |  Features: {list(X.columns)}")
print(f"y shape : {y.shape}")
print()

# Safety assertions
assert "Class" not in X.columns,       "LEAKAGE: Class found in X!"
assert y.nunique() == 2,               "Target must be binary."
assert set(y.unique()) == {0, 1},      "Target values must be 0 or 1."
assert X.shape[1] == 32,               "Expected 30 original + 2 engineered features."

print("Class excluded from X  : VERIFIED")
print("y is binary {0, 1}     : VERIFIED")
print("Feature count (32)     : VERIFIED")


---
## 6. Train / Test Split

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = make_train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

print("=" * 60)
print("STRATIFIED TRAIN / TEST SPLIT")
print("=" * 60)
print(f"  Training shape     : {X_train_raw.shape}")
print(f"  Test shape         : {X_test_raw.shape}")
print()
print(f"  Fraud count — TRAIN : {y_train.sum():,}  ({y_train.mean()*100:.4f}%)")
print(f"  Fraud count — TEST  : {y_test.sum():,}  ({y_test.mean()*100:.4f}%)")
print()

# Stratification verification
overall_rate = y.mean()
train_rate   = y_train.mean()
test_rate    = y_test.mean()
print(f"  Overall fraud rate : {overall_rate*100:.4f}%")
print(f"  Train  fraud rate  : {train_rate*100:.4f}%")
print(f"  Test   fraud rate  : {test_rate*100:.4f}%")

assert abs(train_rate - overall_rate) < 0.001, "Stratification failed — train!"
assert abs(test_rate  - overall_rate) < 0.001, "Stratification failed — test!"

print()
print("Stratification check : PASSED")
print()
print("The test set is now SEALED.")
print("It will NOT be used for any model selection or threshold tuning.")


---
## 7. Feature Engineering — Verification

In [ ]:
print("=" * 60)
print("ENGINEERED FEATURES — VERIFICATION")
print("=" * 60)

# ── Log_Amount ────────────────────────────────────────────────────────────────
print()
print("[1] Log_Amount = log1p(Amount)")
print("    Purpose   : Corrects right-skew of Amount (Phase 1 Task 4).")
print("    Available : Yes — Amount is a real-time transaction field.")
print("    Leakage   : None — not derived from Class.")
la = X_train_raw["Log_Amount"]
print(f"    Train stats: mean={la.mean():.4f}  std={la.std():.4f}  "
      f"min={la.min():.4f}  max={la.max():.4f}")

# ── Hour ──────────────────────────────────────────────────────────────────────
print()
print("[2] Hour = (Time % 86400) / 3600")
print("    Purpose   : Diurnal pattern — genuine txns peak during day,")
print("                fraud is more uniformly distributed (Phase 1 Task 5).")
print("    Available : Yes — Time is a real-time transaction field.")
print("    Leakage   : None — not derived from Class.")
h = X_train_raw["Hour"]
print(f"    Train range: {h.min():.3f} – {h.max():.3f}  (expected 0–23.99)")

# Validity checks
assert X_train_raw["Hour"].max() < 24,        "Hour exceeds 24!"
assert X_train_raw["Hour"].min() >= 0,        "Hour below 0!"
assert X_train_raw["Log_Amount"].min() >= 0,  "Log_Amount below 0!"
assert "Class" not in X_train_raw.columns,    "Class leaked into X_train!"

print()
print("Engineered feature validity checks : PASSED")


In [ ]:
# Visualise the engineered features on training data
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Engineered Features — Training Set: Genuine vs Fraud",
             fontsize=13, fontweight="bold")

df_train_plot = X_train_raw[["Log_Amount", "Hour"]].copy()
df_train_plot["Class"] = y_train.values
gen = df_train_plot[df_train_plot["Class"] == 0]
frd = df_train_plot[df_train_plot["Class"] == 1]

ax = axes[0]
ax.hist(gen["Log_Amount"], bins=70, alpha=0.55, color="#4A90D9",
        label="Genuine", density=True)
ax.hist(frd["Log_Amount"], bins=70, alpha=0.75, color="#E05252",
        label="Fraud", density=True)
ax.set_xlabel("Log_Amount  [log1p(Amount)]")
ax.set_ylabel("Density")
ax.set_title("Log_Amount by Class")
ax.legend()

ax = axes[1]
ax.hist(gen["Hour"], bins=48, alpha=0.55, color="#4A90D9",
        label="Genuine", density=True)
ax.hist(frd["Hour"], bins=48, alpha=0.75, color="#E05252",
        label="Fraud", density=True)
ax.set_xlabel("Hour of Day  [(Time % 86400) / 3600]")
ax.set_ylabel("Density")
ax.set_title("Hour by Class")
ax.legend()

plt.tight_layout()
plt.show()
print("Hour: Genuine transactions show a clear daytime peak; fraud is flatter.")
print("Log_Amount: Fraud transactions cluster at lower amounts but with some high-end extremes.")


---
## 8. Preprocessing & Scaling

In [ ]:
print("=" * 60)
print("SCALING STRATEGY")
print("=" * 60)
print()
print("  Scaler     : RobustScaler")
print("  Rationale  : Amount is right-skewed with extreme high-value outliers.")
print("               RobustScaler (median + IQR) is resistant to outliers,")
print("               so extreme values are scaled without being discarded.")
print()
print("  Cols scaled:", COLS_TO_SCALE)
print("  Not scaled : V1-V28 (PCA-normalised in source data — already centred)")
print()
print("  CRITICAL: Scaler MUST be fitted on X_train only.")
print("            Fitting on full data or test data = leakage.")


In [ ]:
# Fit scaler on TRAINING DATA ONLY
scaler, cols_scaled = fit_scaler(X_train_raw)

print(f"Scaler fitted on X_train ({len(X_train_raw):,} rows).")
print(f"Columns scaled : {cols_scaled}")
print()
print("Scaler parameters (centre=median, scale=IQR) — from training data:")
for col, centre, scale in zip(cols_scaled, scaler.center_, scaler.scale_):
    print(f"  {col:<14}: median={centre:.6f}   IQR={scale:.6f}")


In [ ]:
# Apply scaler — transform ONLY, never refit
X_train_scaled = apply_scaler(X_train_raw, scaler, cols_scaled)
X_test_scaled  = apply_scaler(X_test_raw,  scaler, cols_scaled)

print("Scaler applied to train and test sets (transform only, no refit).")
print()
print("Scaled column statistics on X_train_scaled:")
print(X_train_scaled[cols_scaled].describe().round(4)
      .T[["mean", "std", "min", "max"]].to_string())
print()

# V-features must be unchanged
for vcol in ["V1", "V14", "V17"]:
    diff = abs(X_train_raw[vcol].mean() - X_train_scaled[vcol].mean())
    assert diff < 1e-10, f"V-feature {vcol} was inadvertently modified!"
print("V1, V14, V17 unchanged after scaling : VERIFIED")


---
## 9. Outlier Strategy & Documentation

In [ ]:
print("=" * 60)
print("OUTLIER STRATEGY")
print("=" * 60)
print()
print("Phase 1 finding: Amount has extreme right-skew.")
print()
print("Amount percentiles (training set):")
for p in [90, 95, 98, 99, 99.5, 99.9]:
    val = X_train_raw["Amount"].quantile(p / 100)
    print(f"  P{p:<5.1f}: ${val:>10,.2f}")
print(f"  Max   : ${X_train_raw['Amount'].max():>10,.2f}")
print()
print("Decision: PRESERVE original values by default.")
print("Rationale:")
print("  - Extreme Amount values may represent genuine fraud signals.")
print("  - Auto-removal risks discarding discriminative information.")
print("  - Winsorisation will be tested as an experiment in Phase 3.")
print()

# Compute thresholds from training data ONLY (stored for Phase 3 use)
thresholds = get_outlier_thresholds(X_train_raw,
                                    cols=["Amount", "Log_Amount"],
                                    lower_pct=1.0,
                                    upper_pct=99.0)
print("Optional winsorisation thresholds (training-derived — NOT applied by default):")
for col, bounds in thresholds.items():
    print(f"  {col:<14}: lower={bounds['lower']:.4f}   upper={bounds['upper']:.4f}")
print()
print("These thresholds are available for Phase 3 outlier-treatment experiments.")


---
## 10. Leakage Checks

In [ ]:
print("=" * 60)
print("DATA LEAKAGE CHECKS")
print("=" * 60)
print()

results = {}

# [1] Class not in X
results["[1] Class not in X"] = "Class" not in X_train_scaled.columns

# [2] Scaler fitted on train data only — centre should equal training median
train_medians = {c: X_train_raw[c].median() for c in cols_scaled}
scaler_centres = dict(zip(cols_scaled, scaler.center_))
all_close = all(
    abs(scaler_centres[c] - train_medians[c]) < 1e-6 for c in cols_scaled
)
results["[2] Scaler fitted on train data only"] = all_close

# [3] No SMOTE applied
results["[3] No SMOTE applied in this phase"] = True

# [4] Scaler not fitted on full dataset (train + test medians differ from full)
full_median_amount  = X["Amount"].median()
train_median_amount = X_train_raw["Amount"].median()
# They may or may not differ, but we verify the scaler centre equals TRAIN median
results["[4] Scaler centre = train median (not full dataset)"] = all_close

# [5] No target-derived feature in X
no_target_in_feats = not any(
    "class" in c.lower() or c == TARGET_COL for c in X_train_scaled.columns
)
results["[5] No target-derived feature in X"] = no_target_in_feats

# [6] No duplicates remain that could split across train/test
post_dedup_dups = df_clean.duplicated().sum()
results["[6] No duplicates after dedup (leakage-safe split)"] = (post_dedup_dups == 0)

# [7] Test-set fraud rate within tolerance of overall
results["[7] Test fraud rate preserved (stratification)"] = (
    abs(y_test.mean() - y.mean()) < 0.001
)

# Print
all_passed = True
for check, passed in results.items():
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"  [{status}] {check}")

print()
if all_passed:
    print("All 7 leakage checks PASSED. Pipeline is clean and safe.")
else:
    raise RuntimeError("One or more leakage checks FAILED. Review above.")


---
## 11. Final Dataset Summary

In [ ]:
print("=" * 65)
print("PHASE 2 — FINAL PREPROCESSING SUMMARY")
print("=" * 65)
print()
print(f"DATASET")
print(f"  Original row count        : {ORIGINAL_SHAPE[0]:,}")
print(f"  Rows removed (exact dups) : {n_removed:,}")
print(f"  Rows after dedup          : {len(df_clean):,}")
print(f"  Predictor columns         : {X_train_scaled.shape[1]}  (30 original + 2 engineered)")
print(f"  Engineered features       : ['Log_Amount', 'Hour']")
print()
print(f"TRAIN / TEST SPLIT  (stratified, random_state={RANDOM_SEED})")
print(f"  Training rows             : {len(X_train_scaled):,}")
print(f"  Test rows                 : {len(X_test_scaled):,}")
print(f"  Fraud in training         : {y_train.sum():,}  ({y_train.mean()*100:.4f}%)")
print(f"  Fraud in test             : {y_test.sum():,}  ({y_test.mean()*100:.4f}%)")
print()
print(f"PREPROCESSING DECISIONS")
print(f"  Scaler                    : RobustScaler (median + IQR)")
print(f"  Columns scaled            : {cols_scaled}")
print(f"  V1-V28                    : NOT rescaled (already PCA-normalised)")
print(f"  Scaler fitted on          : Training data only")
print(f"  SMOTE                     : NOT applied (Phase 3 model training only)")
print()
print(f"OUTLIER TREATMENT")
print(f"  Default action            : Preserve original values")
print(f"  Winsorisation thresholds  : Computed from training data (stored)")
print(f"  Winsorisation applied     : NO (optional Phase 3 experiment)")
print()
print(f"LEAKAGE CHECKS")
print(f"  All 7 checks              : PASSED")
print()
print(f"FILES CREATED THIS PHASE")
print(f"  notebooks/02_preprocessing_feature_engineering.ipynb")
print(f"  src/preprocessing.py")


---
## Phase 2 Conclusion

### Actions Taken

| Step | Action | Outcome |
|------|--------|---------|
| 1 | Reload raw CSV & verify integrity | Shape confirmed; CSV untouched |
| 2 | Audit duplicates & check for label conflicts | Conflict check performed; decision documented |
| 3 | Drop exact duplicates before split | Non-first occurrences removed |
| 4 | Engineer features: `Log_Amount`, `Hour` | 2 new features; no target leakage |
| 5 | Separate X / y | `Class` excluded from predictors |
| 6 | Stratified 80/20 train/test split | Fraud rate preserved in both sets |
| 7 | Fit `RobustScaler` on `X_train` only | No test-set leakage |
| 8 | Transform `X_train` and `X_test` | V1–V28 unchanged; Amount/Time/engineered scaled |
| 9 | Compute outlier thresholds from train | Stored for Phase 3 optional experiment |
| 10 | 7-point leakage check | All checks PASSED |

### What is Available for Phase 3

- `X_train_scaled`, `X_test_scaled` — scaled predictor matrices (regenerated from CSV)
- `y_train`, `y_test` — binary target vectors
- `scaler` — fitted `RobustScaler` for new/future data transformation
- `thresholds` — percentile-based bounds for optional winsorisation experiment
- [`src/preprocessing.py`](../src/preprocessing.py) — reusable module (model-agnostic)

### Key Decisions for Phase 3

1. **Primary metric:** Average Precision (PR-AUC) — handles severe class imbalance correctly.
2. **Imbalance handling:** `class_weight='balanced'` in models; SMOTE applied inside CV on training folds only.
3. **Outlier experiment:** Compare baseline vs winsorised — report impact on held-out fraud recall.
4. **Test set:** Completely sealed until final reporting. Zero selection based on test performance.
5. **Models to evaluate:** Logistic Regression → Random Forest → XGBoost → LightGBM → Stacking.

---
*Phase 2 complete. Proceed to `03_modelling.ipynb` in Phase 3.*